# Cover figure for *Paleoceanography and Paleoclimatology*

This notebook draws the cover design for the TEXAS paper (Rattanasriampaipong, Tierney, Elling & Inglis, 2026): the TEXAS calibration curve on a black ground, the published TEX86 calibrations it is compared with, and two sketches of the nonthermal effects, (A) nutrients and (B) depth ecology. It reproduces the hand-drawn reference `figures/cover/cover_PALO_sst_revised.png`.

The curves are real: the TEXAS curve and its two shifted versions come from the default posterior, and the linear (Schouten et al., 2002), TEX86-H (Kim et al., 2010) and BAYSPAR (Tierney & Tingley, 2014) curves are evaluated as published. Labels, ring symbols and the two sketches are illustration, placed by hand.

Nothing here refits a calibration or runs Stan. The notebook only reads a saved posterior and a spreadsheet that are already in the repository, so it runs in a few seconds.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import logging
from matplotlib.font_manager import FontProperties
from matplotlib.patches import (Circle, FancyArrowPatch, PathPatch, Polygon, Rectangle,
                                RegularPolygon)
from matplotlib.textpath import TextPath
from matplotlib.transforms import Affine2D

from TEXAS.stan.io import load_posterior
from TEXAS.models.logistics import generalized_logistic_fixed_upper
from TEXAS.models.calibration import CalibrationRegistry


def find_repo_root(start=None):
    """Walk up from the working directory to the folder holding pyproject.toml."""
    here = Path(start or Path.cwd()).resolve()
    for p in (here, *here.parents):
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("Run this notebook from inside the TEXAS repository.")


REPO = find_repo_root()
SHEETS = REPO / "data" / "spreadsheets"
DATA_CSV = SHEETS / "ds_gridded_screened_global_compilation_finalized.csv"
OUT_DIR = REPO / "figures" / "cover"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Switches ────────────────────────────────────────────────────────────────
# Default TEXAS calibration (temperature + G23 + nitrate). "sst" = sea surface
# temperature target, "thm" = thermocline-integrated temperature (Thermo-T).
TARGET = "sst"
CASE = {"sst": "tx.GHEB.sst.sri03.G23-N1p0", "thm": "tx.GHEB.thm.sri03.G23-N1p0"}[TARGET]
T_COLUMN = {"sst": "SST", "thm": "t_sf2tc_avg"}[TARGET]
Y_COLUMN = "scaledRI_cren3"
NO3_COLUMN = "no3_sf2tc_avg"

# BAYSPAR is drawn for three regions with contrasting local calibrations, the
# same three as Fig. 1 of the paper: compilation region name -> label on the image.
BAYSPAR_REGIONS = {
    "Red Sea": "Red Sea",
    "Mediterranean": "Mediterranean",
    "Pacific-Southern Ocean": "Southern Ocean",
}

SEED = 2026          # fixes the random dots and BAYSPAR draws, so the image is reproducible
T_GRID = np.linspace(-3, 50, 531)

# ── Layout switches ─────────────────────────────────────────────────────────
ARC_TOP = True         # cut the top edge to the curved outline of the cover template
SHOW_WORDMARK = True   # TEXAS tiles and tagline in the lower right
FIG_WIDTH_IN = 11.5    # figure width in inches; the height follows the canvas shape
# Scaled RI at which the (A) and (B) arrows are drawn between the curves.
ARROW_LEVEL_A, ARROW_LEVEL_B = 0.714, 0.736

# ── Colors ──────────────────────────────────────────────────────────────────
C_BACKGROUND = "#0A0A0A"
C_TEXT = "#FFFFFF"
C_TEXAS = "#E1F5EE"
C_LINEAR, C_TEXH, C_BAYSPAR = "#C92A2A", "#E67700", "#862E9C"
C_NUTRIENT = "#9FE1CB"                       # nutrient dots
C_CELL_NUTRIENT = "#0B1F33"                  # cells in sketch (A)
C_CELL_SHALLOW, C_CELL_DEEP = "#782121", "#A0892C"   # cells in sketch (B)
# Wordmark tiles T, E, X, A, S. The documentation site uses the same five colors
# (docs/_static/custom.css and texas_logo.svg); change them together.
C_TILES = ("#B63D3D", "#AC753A", "#7A4486", "#3D6386", "#245058")

# ── Fonts ───────────────────────────────────────────────────────────────────
# The reference drawing uses TeX Gyre Heros (a Helvetica look-alike). The first
# font in this list that is installed is used; put your choice first.
FONT_FAMILY = ["TeX Gyre Heros", "Helvetica", "Arial", "Nimbus Sans", "Liberation Sans",
               "DejaVu Sans"]
FONT_SCALE = 1.0       # multiplies every size below
FONT_SIZES = {         # points
    "label": 12.8,     # sketch labels and the cool / warm notes
    "effect": 13.5,    # "nutrient effect", "ecology effect"
    "tag": 17,         # circled A and B
    "curve": 21,       # TEX86-H, Linear, BAYSPAR
    "texas": 24,       # TEXAS on the curve
    "tagline": 14.2,   # line under the wordmark
}
TILE_LETTER_HEIGHT = 100   # height of the wordmark letters, in canvas pixels (tiles are 132)
FONT_SIZES = {key: size * FONT_SCALE for key, size in FONT_SIZES.items()}

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # skip missing fonts quietly
mpl.rcParams.update({
    "font.family": FONT_FAMILY,
    "mathtext.default": "bf",        # subscripts follow the bold labels
    "svg.fonttype": "none",          # keep text as editable text in SVG ...
    "pdf.fonttype": 42,              # ... and in PDF
})


/sessions/rcw-019thvzosovgfod7gqg9scuy/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/sessions/rcw-019thvzosovgfod7gqg9scuy/mnt/TEXAS/src/TEXAS/utils/paths.py:182: UserWarning: CmdStan not found — Stan sampling (forward calibration and inverse reconstruction) will not be available until CmdStan is installed.
  Install : TEXAS.install_cmdstan()   (one call; also repairs a broken install)
  Diagnose: texas-doctor   (or TEXAS.doctor())
  warnings.warn(


## Data and posterior

The spreadsheet is the screened, gridded compilation used for every calibration in the paper. Coretops are plotted against the modern ocean temperature at each site; cultures and mesocosms are plotted against the temperature they were incubated at.

In [2]:
df = pd.read_csv(DATA_CSV, low_memory=False)

coretop = df[df["datatype"] == "coretop"].dropna(subset=[T_COLUMN, Y_COLUMN])
# Same selection as the culture + mesocosm panel in the paper (Fig. 6).
culmeso = df[df["datatype"].isin(["culture", "mesocosm"])].dropna(subset=["scaledRI", "SST"])
culture = culmeso[culmeso["datatype"] == "culture"]
mesocosm = culmeso[culmeso["datatype"] == "mesocosm"]

post = load_posterior(CASE)
names = {"t0": "t0_crtp", "k": "k_crtp", "b": "b_crtp", "v": "v_crtp",
         "gamma_G23": "gamma_G23_crtp", "gamma_NO3": "gamma_NO3_crtp"}
draws = {key: post[var].values.ravel() for key, var in names.items()}
n_draws = draws["t0"].size
NO3_CUTOFF = float(post.attrs["no3_cutoff"])

print(f"calibration case : {CASE}")
print(f"posterior draws  : {n_draws}")
print(f"coretops         : {len(coretop)}  ({coretop[T_COLUMN].min():.1f} to {coretop[T_COLUMN].max():.1f} °C)")
print(f"cultures         : {len(culture)}  ({culture['SST'].min():.0f} to {culture['SST'].max():.0f} °C)")
print(f"mesocosms        : {len(mesocosm)}  ({mesocosm['SST'].min():.0f} to {mesocosm['SST'].max():.0f} °C)")


calibration case : tx.GHEB.sst.sri03.G23-N1p0
posterior draws  : 4000
coretops         : 1518  (-1.7 to 29.8 °C)
cultures         : 59  (10 to 47 °C)
mesocosms        : 8  (10 to 40 °C)


## TEXAS curve and its two nonthermal shifts

In the default calibration the nonthermal predictors move the curve's location along the temperature axis without changing its shape (paper Eq. 13): `T0_eff = T0 + gamma_G23 * G23 + gamma_NO3 * log10(NO3)`, with the nitrate term switched on only below the cutoff. The main curve is drawn for a reference site: the median coretop G23 and nitrate at or above the cutoff (nitrate term off). The two shifted curves change one predictor at a time, to the 95th percentile of coretop G23 and to the 5th percentile of coretop nitrate.

These are curves for stated predictor values, drawn to show the direction and size of each shift. They are not the in-sample fit to every coretop, which uses each site's own G23 and nitrate.

In [3]:
G23_REF = float(coretop["gdgt23ratio"].median())
G23_HIGH = float(coretop["gdgt23ratio"].quantile(0.95))
NO3_LOW = float(coretop[NO3_COLUMN].quantile(0.05))
assert NO3_LOW < NO3_CUTOFF, "the low-nitrate example must sit below the cutoff"


def texas_curves(t0_shift):
    """One curve per posterior draw (rows = draws, columns = temperatures)."""
    t0_eff = draws["t0"] + draws["gamma_G23"] * G23_REF + t0_shift
    return np.stack([
        generalized_logistic_fixed_upper(T_GRID, t0=t0, k=k, b=b, v=v)
        for t0, k, b, v in zip(t0_eff, draws["k"], draws["b"], draws["v"])
    ])


shift_G23 = draws["gamma_G23"] * (G23_HIGH - G23_REF)        # °C, per draw
shift_NO3 = draws["gamma_NO3"] * np.log10(NO3_LOW)           # °C, per draw

curves = texas_curves(0.0)
median_curve = np.median(curves, axis=0)
median_high_G23 = np.median(texas_curves(shift_G23), axis=0)
median_low_NO3 = np.median(texas_curves(shift_NO3), axis=0)



def summary(x):
    lo, mid, hi = np.percentile(x, [2.5, 50, 97.5])
    return f"{mid:+.1f} °C (95% credible interval {lo:+.1f} to {hi:+.1f})"


print(f"reference site      : G23 = {G23_REF:.1f}, nitrate >= {NO3_CUTOFF:g} µmol/L")
print(f"high G23 ({G23_HIGH:.1f})     : curve moves {summary(shift_G23)}")
print(f"low nitrate ({NO3_LOW:.2f}) : curve moves {summary(shift_NO3)}")
print(f"TEXAS median at {T_GRID[-1]:.0f} °C: {median_curve[-1]:.3f}")


reference site      : G23 = 4.4, nitrate >= 1 µmol/L
high G23 (12.0)     : curve moves +4.8 °C (95% credible interval +4.2 to +5.4)
low nitrate (0.03) : curve moves -4.3 °C (95% credible interval -5.0 to -3.6)
TEXAS median at 50 °C: 0.995


## Published calibrations

The three published calibrations predict TEX86, not Scaled RI, so they need one extra step to share this axis. Each is first evaluated as TEX86 against temperature, exactly as published, and the TEX86 values are then converted to Scaled RI with a straight-line relation fitted to the calibration data in this compilation. That conversion is approximate (the two indices are correlated but not interchangeable; the cell prints the correlation and scatter), so these curves show where each calibration runs and where it ends, and should not be read to the third decimal.

The linear and exponential curves stop where they reach TEX86 = 1, the largest value TEX86 can take; beyond that temperature they have no valid prediction. BAYSPAR varies its slope and intercept from place to place and has no single curve. Following Fig. 1 of the paper, it is drawn as three short segments, one each for the Red Sea, the Mediterranean and the Pacific sector of the Southern Ocean: BAYSPAR's median predicted TEX86 at the central coretop location of each region, over that region's coretop temperature range plus 2 °C on either side. This cell needs a BAYSPAR implementation (the `baysparpy` recode or the `bayspar` package from brews/baysparpy); without one, the BAYSPAR segments are skipped and everything else still draws.

In [4]:
both = df.dropna(subset=["TEX86", Y_COLUMN])
conv_slope, conv_icpt = np.polyfit(both["TEX86"], both[Y_COLUMN], 1)
conv_r = np.corrcoef(both["TEX86"], both[Y_COLUMN])[0, 1]
conv_rmse = float(np.std(both[Y_COLUMN] - (conv_icpt + conv_slope * both["TEX86"])))


def tex_to_scaled_ri(tex):
    return conv_icpt + conv_slope * np.asarray(tex, dtype=float)


def published_curve(name):
    """Scaled RI along T_GRID for a registry calibration, cut where TEX86 passes 1."""
    tex = CalibrationRegistry.get(name).predict_tex86(T_GRID)
    valid = tex <= 1.0
    T_end = float(T_GRID[valid][-1]) if not valid.all() else None
    return np.where(valid, tex_to_scaled_ri(tex), np.nan), T_end


linear_curve, T_end_linear = published_curve("schouten02")
texh_curve, T_end_texh = published_curve("kim2010_tex86H")

# BAYSPAR: forward prediction (temperature in, TEX86 out) at one central site per region.
try:
    import baysparpy as _bsr                      # R. Rattanasriampaipong's recode
    _ensemble = lambda res: np.asarray(res.tex86)
    BAYSPAR_IMPL = "baysparpy (recode)"
except ImportError:
    try:
        import bayspar as _bsr                    # brews/baysparpy
        _ensemble = lambda res: np.asarray(res.ensemble)
        BAYSPAR_IMPL = "bayspar (brews/baysparpy)"
    except ImportError:
        _bsr, BAYSPAR_IMPL = None, None

bayspar_segments = []      # (label, temperatures, Scaled RI)
if _bsr is not None:
    np.random.seed(SEED)   # BAYSPAR draws random samples; fix them for a reproducible image
    for region, label in BAYSPAR_REGIONS.items():
        sites = coretop[coretop["regionName"] == region]
        lat = float(sites["match_lat_04deg"].median())
        lon = float(sites["match_lon_04deg"].median())
        T_seg = np.linspace(max(sites["SST"].min() - 2, T_GRID[0]), sites["SST"].max() + 2, 50)
        tex = np.percentile(_ensemble(_bsr.predict_tex(T_seg, lat=lat, lon=lon, temptype="sst")),
                            50, axis=1)
        # Straight segment between the end points, as in Fig. 1 of the paper.
        bayspar_segments.append((label, T_seg[[0, -1]], tex_to_scaled_ri(tex[[0, -1]])))
        slope = (tex[-1] - tex[0]) / (T_seg[-1] - T_seg[0])
        print(f"BAYSPAR {label:15s}: {len(sites):3d} coretops, centre {lat:.1f}°N {lon:.1f}°E, "
              f"{T_seg[0]:.1f} to {T_seg[-1]:.1f} °C, TEX86 {tex[0]:.3f} to {tex[-1]:.3f} "
              f"({slope:.4f} per °C)")
    print(f"BAYSPAR implementation: {BAYSPAR_IMPL}")
else:
    print("BAYSPAR not installed: the BAYSPAR segments are skipped.")

print(f"TEX86 to Scaled RI: Scaled RI = {conv_icpt:.3f} + {conv_slope:.3f} x TEX86   "
      f"(n = {len(both)}, r = {conv_r:.2f}, scatter = {conv_rmse:.3f})")
print(f"linear (Schouten et al., 2002) reaches TEX86 = 1 at {T_end_linear:.1f} °C")
print(f"TEX86-H (Kim et al., 2010) reaches TEX86 = 1 at {T_end_texh:.1f} °C")


BAYSPAR Red Sea        :  22 coretops, centre 22.6°N 37.6°E, 23.4 to 31.6 °C, TEX86 0.762 to 0.923 (0.0194 per °C)
BAYSPAR Mediterranean  : 122 coretops, centre 41.9°N 12.8°E, 14.5 to 24.8 °C, TEX86 0.484 to 0.618 (0.0130 per °C)
BAYSPAR Southern Ocean :  30 coretops, centre -61.9°N -158.0°E, -3.0 to 10.3 °C, TEX86 0.328 to 0.454 (0.0095 per °C)
BAYSPAR implementation: bayspar (brews/baysparpy)
TEX86 to Scaled RI: Scaled RI = 0.155 + 0.812 x TEX86   (n = 1585, r = 0.90, scatter = 0.053)
linear (Schouten et al., 2002) reaches TEX86 = 1 at 48.0 °C
TEX86-H (Kim et al., 2010) reaches TEX86 = 1 at 38.6 °C


## Drawing

The design is laid out on a 2240 x 1651 canvas, the pixel size of the reference drawing `figures/cover/cover_PALO_sst_revised.png`, so every position below can be read off that image. The curves are the exception: they are plotted in temperature and Scaled RI and come straight from the posterior and the published calibrations, so they move if the calibration changes while the labels and the two mechanism sketches stay where they are placed.

Positions, angles and wording are plain numbers and strings inside `draw_design`; edit them there. Colors, fonts and line widths are set in the first code cell.

In [5]:
CANVAS_W, CANVAS_H = 2240, 1651

# Temperature and Scaled RI limits that put the TEXAS curve where the design has it:
# -3 °C at canvas x = 270 and 50 °C at x = 1675; Scaled RI 0.447 at canvas y = 1287 and
# 0.995 at y = 338.
_PX_PER_DEGC = (1675 - 270) / (50 - (-3))
_PX_PER_RI = (1287 - 338) / (0.995 - 0.447)
T_LIMITS = (-3 - 270 / _PX_PER_DEGC, -3 + (CANVAS_W - 270) / _PX_PER_DEGC)
RI_LIMITS = (0.447 - (CANVAS_H - 1287) / _PX_PER_RI, 0.447 + 1287 / _PX_PER_RI)


def _pt(px):
    """Canvas pixels to points at the figure size used below."""
    return px * FIG_WIDTH_IN * 72 / CANVAS_W


def _ring(ax, x, y, radius=24, dashed=False):
    """One cyclopentane ring, drawn as an outlined pentagon."""
    ax.add_patch(RegularPolygon((x, y), 5, radius=radius, orientation=np.pi, facecolor="none",
                                edgecolor=C_TEXT, linewidth=_pt(4.2),
                                linestyle=(0, (1.1, 1.1)) if dashed else "-", joinstyle="round"))


def _cell(ax, x, y, width, height, fill, outline=3.0):
    """An archaeal cell, drawn as a capsule."""
    r = height / 2
    turn = np.linspace(-np.pi / 2, np.pi / 2, 40)
    xs = np.r_[x + width / 2 - r + r * np.cos(turn), x - width / 2 + r - r * np.cos(turn)]
    ys = np.r_[y + r * np.sin(turn), y - r * np.sin(turn)]
    ax.add_patch(Polygon(np.column_stack([xs, ys]), closed=True, facecolor=fill,
                         edgecolor=C_TEXT, linewidth=_pt(outline), joinstyle="round"))


def _tag(ax, x, y, letter):
    """A circled letter linking a curve shift to its mechanism sketch."""
    ax.add_patch(Circle((x, y), 30, facecolor="none", edgecolor=C_TEXT, linewidth=_pt(3.2)))
    ax.text(x, y + 1, letter, color=C_TEXT, fontsize=FONT_SIZES["tag"], fontweight="bold",
            ha="center", va="center")


def _arrow(ax, start, end, dashed=False):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=_pt(38),
                                 color=C_TEXT, linewidth=_pt(4.0), shrinkA=0, shrinkB=0,
                                 linestyle=(0, (1.6, 1.6)) if dashed else "-"))


def _tile_letter(ax, x, y, letter):
    """A wordmark letter centered on (x, y) by its drawn outline.

    Ordinary text is centered on the font's line box, which leaves capitals sitting
    high and letters such as T and S off to one side. Here the letter is turned into
    an outline and the outline's own bounding box is centered, so every letter sits
    in the middle of its tile. The letters are therefore shapes, not editable text.
    """
    font = FontProperties(family=FONT_FAMILY, weight="bold")
    scale = TILE_LETTER_HEIGHT / TextPath((0, 0), "E", size=100, prop=font).get_extents().height
    outline = TextPath((0, 0), letter, size=100, prop=font)
    box = outline.get_extents()
    place = (Affine2D().translate(-(box.x0 + box.x1) / 2, -(box.y0 + box.y1) / 2)
             .scale(scale, -scale).translate(x, y))          # y runs downward on the canvas
    ax.add_patch(PathPatch(outline, facecolor=C_TEXT, edgecolor="none",
                           transform=place + ax.transData))


def _label(ax, x, y, text, size="label", bold=True, **kwargs):
    kwargs.setdefault("ha", "center")
    kwargs.setdefault("va", "center")
    kwargs.setdefault("color", C_TEXT)
    ax.text(x, y, text, fontsize=FONT_SIZES[size], fontweight="bold" if bold else "normal",
            **kwargs)


def draw_curves(ax):
    """Everything that is data: TEXAS, its two shifted curves, and the published curves."""
    ax.plot(T_GRID, linear_curve, color=C_LINEAR, lw=_pt(4.5), solid_capstyle="butt")
    ax.plot(T_GRID, texh_curve, color=C_TEXH, lw=_pt(4.5), solid_capstyle="butt")
    for _, T_seg, ri_seg in bayspar_segments:
        ax.plot(T_seg, ri_seg, color=C_BAYSPAR, lw=_pt(6.0), solid_capstyle="butt")

    ax.plot(T_GRID, median_low_NO3, color=C_TEXT, lw=_pt(2.2), ls=(0, (1.6, 1.6)))   # (A)
    ax.plot(T_GRID, median_high_G23, color=C_TEXT, lw=_pt(1.3), alpha=0.85)          # (B)
    ax.plot(T_GRID, median_curve, color=C_TEXAS, lw=_pt(7.5), solid_capstyle="round")

    # Arrows from the TEXAS curve to each shifted curve, at a fixed Scaled RI.
    for shifted, level in ((median_low_NO3, ARROW_LEVEL_A), (median_high_G23, ARROW_LEVEL_B)):
        T_from = np.interp(level, median_curve, T_GRID)
        T_to = np.interp(level, shifted, T_GRID)
        ax.annotate("", xy=(T_to, level), xytext=(T_from, level),
                    arrowprops=dict(arrowstyle="-|>", color=C_TEXT, lw=_pt(3.2),
                                    shrinkA=0, shrinkB=0, mutation_scale=_pt(34)))


def draw_design(*, arc_top=ARC_TOP, wordmark=SHOW_WORDMARK):
    """Draw the cover design and return the figure.

    arc_top  : cut the top edge to the curved outline of the journal's cover template.
    wordmark : draw the TEXAS tiles and the tagline in the lower right.
    """
    fig = plt.figure(figsize=(FIG_WIDTH_IN, FIG_WIDTH_IN * CANVAS_H / CANVAS_W))

    # Layer 1: curves, in temperature and Scaled RI.
    data = fig.add_axes([0, 0, 1, 1], facecolor=C_BACKGROUND)
    data.set_xlim(*T_LIMITS)
    data.set_ylim(*RI_LIMITS)
    data.axis("off")
    data.add_patch(Rectangle((T_LIMITS[0], RI_LIMITS[0]), T_LIMITS[1] - T_LIMITS[0],
                             RI_LIMITS[1] - RI_LIMITS[0], facecolor=C_BACKGROUND, zorder=0))
    draw_curves(data)

    # Layer 2: labels and sketches, in canvas pixels (y runs downward, as in the image).
    ax = fig.add_axes([0, 0, 1, 1], facecolor="none")
    ax.set_xlim(0, CANVAS_W)
    ax.set_ylim(CANVAS_H, 0)
    ax.axis("off")

    # ── Curve names ────────────────────────────────────────────────────────
    _label(ax, 1640, 292, "TEXAS", "texas")
    _label(ax, 1372, 312, "TEX$_{86}^{H}$", "curve", color=C_TEXH, rotation=61.4,
           rotation_mode="anchor")
    _label(ax, 1590, 465, "Linear", "curve", color=C_LINEAR, rotation=41.1,
           rotation_mode="anchor")
    if bayspar_segments:
        _label(ax, 1065, 563, "BAYSPAR", "curve", color=C_BAYSPAR, rotation=45.5,
               rotation_mode="anchor")
    _label(ax, 960, 800, "nutrient effect", "effect", rotation=50.3, rotation_mode="anchor")
    _label(ax, 1172, 940, "ecology effect", "effect", rotation=46.1, rotation_mode="anchor")
    _tag(ax, 831, 935, "A")
    _tag(ax, 1301, 803, "B")

    # ── Cool and warm ends of the curve ────────────────────────────────────
    for x in (468, 512):
        _ring(ax, x, 1007, radius=19)
    for i, line in enumerate(("cooler T", "low ring numbers", "low TEX$_{86}$")):
        _label(ax, 534, 1057 + 42 * i, line, ha="right")
    for x in (1858, 1904, 1950):
        _ring(ax, x, 286, radius=19)
    for i, line in enumerate(("warmer T", "high ring numbers", "high TEX$_{86}$")):
        _label(ax, 1835, 347 + 42 * i, line, ha="left")

    # ── (A) Nutrient effect: the same cell with and without nutrients ──────
    _label(ax, 307, 290, "nutrient-rich")
    _label(ax, 579, 290, "nutrient-poor")
    ax.plot([441, 441], [322, 762], color=C_TEXT, lw=_pt(1.6), ls=(0, (3, 3)), alpha=0.7)
    rng = np.random.default_rng(SEED)
    ax.scatter(240 + 175 * rng.random(42), 335 + 265 * rng.random(42), s=_pt(7) ** 2,
               color=C_NUTRIENT, linewidths=0)
    ax.scatter([622, 571, 567, 594, 550], [356, 383, 440, 537, 563], s=_pt(7) ** 2,
               color=C_NUTRIENT, linewidths=0)
    _cell(ax, 340, 460, 102, 44, C_CELL_NUTRIENT, outline=2.0)
    _cell(ax, 556, 461, 104, 46, C_CELL_NUTRIENT, outline=4.5)
    _ring(ax, 340, 730)
    _ring(ax, 503, 730)
    _ring(ax, 574, 730, dashed=True)
    ax.add_patch(FancyArrowPatch((574, 702), (594, 650), arrowstyle="-", color=C_TEXT,
                                 linewidth=_pt(1.6), connectionstyle="arc3,rad=-0.35"))
    _tag(ax, 669, 585, "A")
    _label(ax, 597, 645, "add ring", ha="left")
    _label(ax, 597, 684, "warm bias", bold=False, ha="left")

    # ── (B) Ecology effect: shallow and deep clades both reach the sediment ─
    _label(ax, 1677, 693, "shallow clades")
    _tag(ax, 1934, 686, "B")
    xs = np.linspace(1575, 2008, 200)
    ax.plot(xs, 735 + 6 * np.sin((xs - 1575) / 46), color=C_TEXT, lw=_pt(1.6))
    ax.plot(xs, 822 + 5 * np.sin((xs - 1575) / 52 + 1.0), color=C_TEXT, lw=_pt(1.6),
            ls=(0, (1, 2.2)))
    _cell(ax, 1654, 768, 96, 44, C_CELL_SHALLOW)
    _cell(ax, 1902, 770, 96, 44, C_CELL_SHALLOW)
    _cell(ax, 1967, 855, 92, 40, C_CELL_DEEP, outline=1.6)
    _label(ax, 2022, 898, "deep clades")
    _arrow(ax, (1654, 805), (1654, 972))
    _arrow(ax, (1902, 805), (1902, 980))
    _arrow(ax, (1973, 920), (1973, 982), dashed=True)
    for x in (1620, 1680):
        _ring(ax, x, 1028)
    for x in (1885, 1945):
        _ring(ax, x, 1028)
    _ring(ax, 2003, 1028, radius=22, dashed=True)
    _label(ax, 1650, 1084, "surface signal")
    _label(ax, 1650, 1123, "low 2/3")
    _label(ax, 1942, 1084, "mixed signal")
    _label(ax, 1942, 1123, "high 2/3")
    _label(ax, 1946, 1163, "cold bias", bold=False)

    # ── Wordmark ───────────────────────────────────────────────────────────
    if wordmark:
        for i, (letter, color) in enumerate(zip("TEXAS", C_TILES)):
            x0 = 1259 + 134.2 * i
            ax.add_patch(Rectangle((x0, 1292), 134.2, 132, facecolor=color,
                                   edgecolor="#D9D9D9", linewidth=_pt(1.2)))
            _tile_letter(ax, x0 + 67.1, 1358, letter)
        _label(ax, 1583, 1469, "coretop + culture + mesocosm + nonthermal effects", "tagline")

    # ── Curved top edge of the cover template ──────────────────────────────
    if arc_top:
        radius, x_c = 3985.0, 1158.0
        xs = np.linspace(-5, CANVAS_W + 5, 400)
        ys = radius - np.sqrt(radius ** 2 - (xs - x_c) ** 2)
        ax.add_patch(Polygon(np.column_stack([np.r_[xs, CANVAS_W + 5, -5],
                                              np.r_[ys, -5, -5]]),
                             closed=True, facecolor="white", edgecolor="none", zorder=20))
    return fig


## Render

One image is written to `figures/cover/`, as PDF, SVG and a 300 dpi PNG. Text stays editable in the PDF and SVG. `cover_PALO_sst_revised.*` in the same folder is the hand-drawn reference this notebook reproduces; it is not overwritten.

In [6]:
fig = draw_design()
stem = OUT_DIR / f"cover_PALO_{TARGET}_design"
for ext in ("pdf", "svg", "png"):
    fig.savefig(f"{stem}.{ext}", dpi=300, facecolor="white")
    print("wrote", stem.relative_to(REPO).with_suffix(f".{ext}"))


wrote figures/cover/cover_PALO_sst_design.pdf
wrote figures/cover/cover_PALO_sst_design.svg


wrote figures/cover/cover_PALO_sst_design.png


## Draft cover caption

Cover: Schematic of the TEXAS calibration, which relates the number of rings in archaeal membrane lipids (the Scaled Ring Index) to temperature with an S-shaped curve that levels off at both ends. The white curve is the median TEXAS calibration; colored lines are published TEX86 calibrations for comparison: the original linear calibration (Schouten et al., 2002), TEX86-H (Kim et al., 2010) and BAYSPAR (Tierney & Tingley, 2014) for three regions. Arrows show the two nonthermal effects in TEXAS. (A) Nutrient effect: where nutrients are scarce, archaea add rings, which reads as warmer if uncorrected. (B) Ecology effect: where deeper-dwelling clades contribute lipids, marked by a high GDGT-2/GDGT-3 ratio, the sediment signal reads colder than the surface. See Rattanasriampaipong et al. (https://doi.org/...).